In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
import joblib

print("Libraries loaded")

In [ ]:
df = pd.read_csv("../data/processed/Maths_cleaned.csv")
print(f"Shape: {df.shape}")
print("Data loaded.")

In [ ]:
def categorize_grade(g3):
    if g3 < 10:
        return 'Struggling'
    elif g3 <= 14:
        return 'Average'
    else:
        return 'Excelling'

df['performance'] = df['G3'].apply(categorize_grade)

print("Performance category distribution")
print(df['performance'].value_counts())

#Visualize

plt.figure(figsize = (7,4))
df['performance'].value_counts().plot(kind='bar', color=['blue', 'orange', 'yellow'])
plt.title("Student Performance Categories")
plt.xlabel('Category')
plt.ylabel('Number of Students')
plt.xticks(rotation = 0)
plt.show()

In [ ]:
#Based on the exploration findings
features = [
    'G1', 'G2',      # strongest predictors
    'failures',      # strong negative predictor
    'Medu', 'Fedu',  # parental education
    'higher',        # student motivation to go to higher ed
    'studytime',     # study habits
    'absences',     # attendance
    'goout',         # lifestyle (going out/socializing)
    'Walc', 'Dalc',  # weekend/weekday alcohol consumption respectively
    'age',           # age
    'romantic',      # relationship status
    'traveltime'     # commute
]

X = df[features]
y = df['performance']

print(f"Features shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.2, random_state = 42, stratify = y
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features scaled")

In [ ]:
model = RandomForestClassifier(
    n_estimators = 100,
    random_state = 42
)

model.fit(X_train_scaled, y_train)
print("Model trained")

In [ ]:
y_pred = model.predict(X_test_scaled)

print("Classification Report:")
print(classification_report(y_test, y_pred))

In [ ]:
plt.figure(figsize = (8,6))
cm = confusion_matrix(y_test, y_pred, labels = ['Struggling', 'Average', 'Excelling'])
sns.heatmap(cm, annot = True, fmt = 'd', cmap = 'Blues',
            xticklabels = ['Struggling', 'Average', 'Excelling'],
            yticklabels = ['Struggling', 'Average', 'Excelling'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

In [ ]:
importance_df = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
    }).sort_values('importance', ascending = False)

plt.figure(figsize = (10, 6))
sns.barplot(x = 'importance', y = 'feature', data = importance_df, palette = "Blues_r")
plt.title('Feature Importance')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.show()

print(importance_df)

In [ ]:
joblib.dump(model, '../models/performance_model.pk1')
joblib.dump(scaler, '../models/scaler.pk1')
print("Model and scaler saved")